In [ ]:
import trafilatura
from bs4 import BeautifulSoup

from medici.common.utils.constants import BLOCK_TAGS, HEADING_TAGS, SKIP_TAGS

In [ ]:
html_path = "data/Beautiful_Soup.html"

In [ ]:
with open(html_path, encoding="utf-8") as f:
    html = f.read()

In [ ]:
content = trafilatura.extract(
    html,
    include_tables=True,
    include_links=False,
    include_images=True,
    no_fallback=False,
    output_format="xml",
)

In [ ]:
content

In [ ]:
soup = BeautifulSoup(html, "lxml")

In [ ]:
for tag in soup(list(SKIP_TAGS)):
    tag.decompose()

In [ ]:
nodes, heading_stack = [], []

In [ ]:
def get_clean_text(tag) -> str:
    return " ".join(tag.get_text(separator=" ").split())

In [ ]:
def table_to_text(table_tag) -> str:
    rows_tags = table_tag.find_all(["tr", "row"])

    def is_header_cell(cell) -> bool:
        return cell.name == "th" or cell.get("role") == "head"

    headers = []
    if rows_tags:
        first_cells = rows_tags[0].find_all(["td", "th", "cell"])
        if any(is_header_cell(c) for c in first_cells):
            headers = [c.get_text(strip=True) for c in first_cells]

    out = []
    for row in rows_tags:
        cells = row.find_all(["td", "th", "cell"])
        values = [c.get_text(strip=True) for c in cells]
        if not values:
            continue
        if (
            headers
            and values
            == [
                c.get_text(strip=True)
                for c in row.find_all(["td", "th", "cell"])
                if is_header_cell(c) or True
            ]
            and row is rows_tags[0]
            and any(is_header_cell(c) for c in cells)
        ):
            continue
        if headers and len(headers) == len(values):
            out.append(" | ".join(f"{h}: {v}" for h, v in zip(headers, values, strict=False)))
        else:
            out.append(" | ".join(values))

    return "\n".join(out)

In [ ]:
def node_to_string(nodes: list[dict]) -> str:
    parts = []

    for node in nodes:
        text = node.get("text", "")
        if not text.strip():
            continue  # never emit a hollow/placeholder entry

        node_type = node["type"]

        if node_type == "heading":
            level = max(1, min(6, node.get("level", 1)))
            parts.append(f"{'#' * level} {text}")

        elif node_type == "block":
            breadcrumb = node.get("breadcrumb")
            prefix = f"[{breadcrumb}]\n" if breadcrumb else ""
            parts.append(f"{prefix}{text}")

        elif node_type == "table":
            breadcrumb = node.get("breadcrumb")
            prefix = f"[{breadcrumb}]\n" if breadcrumb else ""
            parts.append(f"{prefix}<table>\n{text}\n</table>")

        else:
            raise ValueError(f"Unhandled node type in _node_to_string: {node_type!r}")

    return "\n\n".join(parts)

In [ ]:
for tag in soup.find_all(True):
    name = tag.name

    if name in HEADING_TAGS:
        level = int(name[1])

        text = get_clean_text(tag)

        heading_stack = [h for h in heading_stack if h["level"] < level]
        heading_stack.append({"level": level, "text": text})

        nodes.append(
            {
                "type": "heading",
                "level": level,
                "text": text,
                "breadcrumb": " > ".join(h["text"] for h in heading_stack[:-1]),
            }
        )
    elif name in BLOCK_TAGS:
        text = get_clean_text(tag)

        if len(text) < 20:
            continue

        nodes.append(
            {
                "type": "block",
                "text": text,
                "breadcrumb": " > ".join(h["text"] for h in heading_stack),
            }
        )

    elif name == "table":
        nodes.append(
            {
                "type": "table",
                "text": table_to_text(tag),
                "breadcrumb": " > ".join(h["text"] for h in heading_stack),
            }
        )

In [ ]:
node_to_string(nodes)